# Chapter 5: Inspect the real structural proposal and admission

**The question.** The Chapter 5 companion (`regional_case`) followed an older LLM proposal: merge two clusters. This notebook follows a fresh, live run of the same step. Again the LLM read the search results and proposed a change to the search rules, and this time it proposed a different change: add a new feature.

**What the LLM proposed.** Multiply two existing features, business-receipt share and cash intensity, into a new one. The new feature goes into the error model that describes customers, so the 4 clusters form differently. The money-laundering model is not retrained, and a weak group is still one cluster.

**How it was tested.** As in the companion, the proposal was checked, the search with the new feature was raced against the old search given the same extra budget (1,600 cluster scores), and both final answers were measured on 200,000 new customers.

**Result.** The new feature helped at one of the three seeds (74101) and made no measurable difference at the other two.

**Evidence.** Run 20260923T220909Z repeated the regional study end to end on two machines. `spark-fe12` reran the search with three new optimizer seeds (74101 to 74103), made one live LLM proposal and drew a fresh test sample of 200,000 new customers. `spark-ef84` ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. The simulated customers and the frozen model are the same as in the companions, but the seeds, proposal and test sample are new, so do not mix these numbers with the companion's. Rerunning this notebook makes no LLM call.

## Setup and source checks

Finds the saved data for this run and checks, by its SHA-256 fingerprint, that the results file is unchanged.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT).as_posix())
read=lambda p: json.loads((R/p).read_text(encoding="utf-8"))
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## The live proposal, its approval and the test

The cell shows three things: the LLM's proposal, the approval record and the comparison on new customers. The assertions confirm that the call completed and that the approval refers to this exact proposal.

In [2]:
call=read('regional/reflection/live_call/response.json')
assert call['status']=='ok'
proposal=read('regional/reflection/response.json')
admission=read('regional/reflection/admission.json')
assert admission['admitted']
assert admission['response_sha256']==hashlib.sha256((R/'regional/reflection/response.json').read_bytes()).hexdigest()
display(proposal)
display(admission)
display(pd.DataFrame(results['comparisons']))

{'parent_specification': 'leaf-output-medoid-single-cell-v1',
 'change_type': 'derived_feature',
 'operation': 'product',
 'operands': ['business_receipt_share', 'cash_intensity_z'],
 'feature_name': 'business_receipt_cash_intensity_product',
 'rationale': "business_receipt_share (fraction of receipts declared as business activity, pre_period) and cash_intensity_z (standardized deviation of cash receipts from the account's own baseline, period_end) form the single largest pairwise fANOVA variance among all reported pairs (3.95e-05, evidence:fanova_pairs), more than double the next pair (cash_intensity_z x counterparty_concentration_z, 1.79e-05). business_receipt_share's own main effect is essentially flat (~1e-4 in magnitude, evidence:feature:business_receipt_share) while cash_intensity_z has the strongest univariate tail effect of any feature (bin_mean_brier rising to 0.042 in its top decile, main_effect 0.031, evidence:feature:cash_intensity_z). A flat marginal effect paired with a d

{'admitted': True,
 'response_sha256': '1b9c4f943eb9f5f47a8e94dac5959a082039dd2afbdb64b7c69b64f265b5a8d0',
 'version': 'leaf-output-medoid-derived-product-v1',
 'parent': 'leaf-output-medoid-single-cell-v1',
 'reviewer': 'Codex source and executability review',
 'reason': 'Product of two existing dimensionless pre-decision fields; finite on discovery data. Auxiliary may refit; base predictor stays fixed and region remains a single cell.',
 'checks': {'feature_metadata': [{'feature': 'business_receipt_share',
    'meaning': 'Declared expected share of receipts from business activity',
    'available': 'pre_period',
    'unit': 'fraction',
    'eligibility': 'Available before the monitoring decision; not a post-investigation field'},
   {'feature': 'cash_intensity_z',
    'meaning': "Standardized deviation of cash receipts from the account's own baseline",
    'available': 'period_end',
    'unit': 'standardized score',
    'eligibility': 'Available before the monitoring decision; not a 

,seed,a,b,contrast_difference,standard_error,simultaneous_ci
0,74101,reflection,continuation,0.012834,0.001590,"[0.008051940751484483, 0.017616414017858083]"
1,74101,twinning,random,0.000000,0.000000,"[0.0, 0.0]"
2,74102,reflection,continuation,-0.001141,0.001139,"[-0.0045669455524634285, 0.0022844287437013173]"
3,74102,twinning,random,0.000000,0.000000,"[0.0, 0.0]"
4,74103,reflection,continuation,-0.000902,0.001287,"[-0.004772999696062933, 0.002968597897497625]"
5,74103,twinning,random,0.006413,0.001018,"[0.003349358407588642, 0.009475817055983117]"


**Read the output.** **In one line: the LLM proposed a new feature; it was approved, and on new customers it helped at one seed out of three.**

**The proposal (first output).** `change_type` is `derived_feature`: multiply `business_receipt_share` by `cash_intensity_z`. The LLM's reasoning: this pair has the largest joint effect in the error model's breakdown (the functional ANOVA of Chapter 3), yet business-receipt share has almost no effect on its own. Trees of depth two struggle to use a feature like that, so handing them the product could expose a weak group they miss. Its `falsifiable_prediction` says what would prove it wrong: no gain beyond the seed-to-seed noise. Among its `alternative_explanations` it names the other option, merging two clusters, as a live competitor.

**The approval (second output).** Approved as version `leaf-output-medoid-derived-product-v1`. Both input features are known before the monitoring decision, and the product is finite on the discovery data. Only the error model is refitted; the money-laundering model stays frozen and a weak group is still one cluster. The reviewer also noted a slip: the LLM called the bins deciles, but the packet used quintiles.

**The comparison (third output).** **Columns.** `seed`: which run. `a` and `b`: the two searches compared; `reflection` is the search with the new feature and `continuation` the old search given the same extra budget. `contrast_difference`: `a` minus `b` on new customers; positive means `a` found a weaker spot. `standard_error` and `simultaneous_ci`: the uncertainty and a range of plausible values.

**Notice.** New feature minus old search is +0.012834 at seed 74101, with a range [0.008052, 0.017616] above zero, and −0.001141 and −0.000902 at the other two seeds, whose ranges contain zero. The `twinning`/`random` rows compare the two ways of starting the search (Chapter 4). They are exactly zero at seeds 74101 and 74102 because both starts ended with the same group; at seed 74103 twinning is ahead by 0.006413. The three seeds share one discovery sample, one proposal and one test sample, so the single win is not a replicated result.